# Sanity check: trajectory balance (GFlowNet) against GRPO on GSM8K

Three runs with the same prompts and budget on Qwen3-0.6B: GRPO as usually run (T2), GRPO with a drift penalty (T3), and trajectory balance. Each run evaluates GSM8K test accuracy before training and every 20 steps.

Kaggle settings: **Accelerator: GPU T4 x2** (the script uses one GPU) and **Internet: on**. Kaggle gives 30 free GPU-hours a week. The runs have not been timed on a T4 yet.

In [ ]:
REPO_URL = "https://github.com/amitkov-sudo/grpo-vs-gfn-self-correction.git"

!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
# Kaggle ships torchao 0.10. Recent PEFT refuses to add LoRA layers when it finds a torchao older
# than 0.16, and this repository does not use torchao, so remove it.
!pip uninstall -q -y torchao

In [ ]:
!python -m pytest -q tests
!python train/train.py --smoke

In [ ]:
COMMON = "--task gsm8k --model Qwen/Qwen3-0.6B --steps 60 --seed 0"
!python train/train.py $COMMON --method grpo
!python train/train.py $COMMON --method grpo --anchor
!python train/train.py $COMMON --method tb

In [ ]:
import glob, json, os
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for path in sorted(glob.glob("outputs/gsm8k-*/log.jsonl")):
    evals = [r["eval"] for r in map(json.loads, open(path)) if "eval" in r]
    run = os.path.basename(os.path.dirname(path))
    ax.plot([e["step"] for e in evals], [e["accuracy"] for e in evals], marker="o", label=run)
    print(run, [(e["step"], e["accuracy_count"]) for e in evals])
ax.set_xlabel("training step")
ax.set_ylabel("GSM8K test accuracy (first 200 problems)")
ax.legend()
plt.show()